# b3_forecast

> Phase 22 (`specs/multiagent/phase22/`, experiment B3): the forecasters and their files. `Effect` (a signed linear combination of arm means on one measure, oriented so + means the design does better, P22-8) and `estimate` (per-seed values paired on seed, or unpaired arm means); the four forecasters of P22-7, `forecastSim` (fitted, prior-only and unfitted: the 95% `bootSeeds` interval over simulator seeds), `forecastPilot` (the pilot's point estimate, arms it lacks from cached Claude rows, every-design-helps' sign marked `fallback` when neither has an arm, P22-17) and `forecastNaive` (every-design-helps); the entry point `forecastExperiment` (the five lines per effect, FR-C4); `writeForecast` (sorted-key JSON, its SHA-256) and `scoreExperiment` (paper2's rule per item and forecaster, counts on the common set and on all items, FR-D1).

In [ ]:
#| default_exp b3_forecast

In [ ]:
#| hide
from nbdev.showdoc import *

## Effects and their estimates

In [ ]:
#| export
import hashlib
import json
import subprocess
from dataclasses import dataclass
from pathlib import Path

import numpy as np

from explRL.colony_report import NAN, _f, bootSeeds

LINES = ('fitted', 'priorOnly', 'unfitted', 'pilotOnly', 'naive')   # the five lines (FR-C4); prior-only is descriptive
SIMS = ('fitted', 'priorOnly', 'unfitted')                         # the simulator lines: 0 on an unrepresented item (P22-9)
SEEDS = tuple(range(3000, 3030))                                   # simulator seeds, = b3_fit.SIM_SEEDS (Part 0's)
THETA_DEFAULT = {}                                                 # no knob set: today's rule agents (b3_agents.isDefault)
RULE = ("simulators: + or - when the 95% bootSeeds interval over seeds (2,000 resamples) excludes 0, else 0; pilot-only: the point "
        "estimate's sign, or naive's marked fallback when an arm is in neither pilot nor prior rows (P22-17); naive: e.naive (P22-8)")


@dataclass(frozen=True)
class Effect:
    """sum of coef x mean(arm) on `measure`, `fault` and size `N` (an int, or a tuple pooled as Stage 1's pairs). Terms: (coef, arm),
    (coef, arm, N) for an arm at another size, (coef, None) for a constant (H18-3's margin); 'arm|fault' is a full level. `naive`:
    the sign meaning "the design is better" (P22-8), None where every-design-helps does not apply (team size, Phase 14)."""
    id: str; hyp: str; measure: str; fault: str; N: object
    terms: tuple
    naive: object = '+'
    represented: bool = True
    note: str = ''


_sizes = lambda e: tuple(e.N) if isinstance(e.N, (tuple, list)) else (e.N,)
_key = lambda r: (r.get('level'), int(_f(r['N'])) if np.isfinite(_f(r.get('N'))) else None)
_terms = lambda e: [(float(t[0]), t[1] and (t[1] if '|' in t[1] else f'{t[1]}|{e.fault}'), t[2] if len(t) > 2 else None)
                    for t in e.terms]          # [(coef, level or None, N or None)]


def armCells(e):
    "The (level, N) cells an effect reads, in term order."
    return list(dict.fromkeys((lv, n or N) for N in _sizes(e) for _, lv, n in _terms(e) if lv))


def estimate(rows, e, seeds=None, paired=True):
    """(dp, mean) from rows levelled '<arm>|<fault>'. Paired: dp {(seed, N): sum coef x value} on seeds with every term (sorted, as
    `colony_report.pairs`). Unpaired (pilot-only): dp {}, mean = sum coef x arm mean, over sizes. nan when an arm is missing."""
    T, ix = _terms(e), {}
    const, A = sum(c for c, lv, _ in T if lv is None), [t for t in T if t[1] is not None]
    for r in rows:
        if 'level' in r and np.isfinite(v := _f(r.get(e.measure))) and (seeds is None or int(_f(r['seed'])) in seeds):
            ix[(r['level'], int(_f(r['N'])), int(_f(r['seed'])))] = v
    if not paired:
        m = [[[v for (l, n2, _), v in ix.items() if (l, n2) == (lv, n or N)] for _, lv, n in A] for N in _sizes(e)]
        return {}, (float(np.mean([const + sum(c * float(np.mean(v)) for (c, _, _), v in zip(A, mN)) for mN in m]))
                    if all(all(mN) for mN in m) else NAN)
    ks = {(s, N): [(lv, n or N, s) for _, lv, n in A] for N in _sizes(e) for s in sorted({k[2] for k in ix})}
    dp = dict(sorted((k, const + sum(c * ix[x] for (c, _, _), x in zip(A, xs))) for k, xs in ks.items() if all(x in ix for x in xs)))
    return dp, float(np.mean(list(dp.values()))) if dp else NAN

## The forecasters

In [ ]:
#| export
_sign = lambda lo, hi: '+' if lo > 0 else '-' if hi < 0 else '0'


def _signed(res, nboot=2000):
    "dict(estimate, ci, sign) from an estimateFn's (dp, mean), or from a frozen interval rule's dict(estimate, ci)."
    est, lo, hi = ((res['estimate'], *res['ci']) if isinstance(res, dict) else
                   bootSeeds(res[0], nboot=nboot) if res[0] else (res[1], NAN, NAN))
    return dict(estimate=est, ci=[lo, hi], sign=_sign(lo, hi))


def forecastSim(effects, cellFn, theta, seeds=SEEDS, estimateFn=estimate, nboot=2000):
    """Fitted, prior-only or unfitted (by `theta`, FR-C3): each represented effect's cells, each (cell, seed) once, through
    `cellFn(arm, fault, N, seed, theta) -> row dict` (one board with the effect's measure; level, N and seed filled in when absent),
    signed by the 95% `bootSeeds` interval. `estimateFn(rows, e)`: (dp, mean), or dict(estimate, ci) for an unpaired rule (P14)."""
    rows, done, out = [], set(), {}
    for e in [e for e in effects if e.represented]:
        for lv, N, s in ((lv, N, s) for lv, N in armCells(e) for s in seeds if (lv, N, s) not in done):
            done.add((lv, N, s))
            rows.append(dict(dict(level=lv, N=N, seed=s), **cellFn(*lv.split('|', 1), N, s, theta)))
        out[e.id] = _signed(estimateFn(rows, e), nboot)
    return out


def forecastPilot(effects, pilotRows, priorRows, estimateFn=estimate):
    """Pilot-only (FR-C3, P22-16, P22-17): each cell from the pilot if it has it, else from `priorRows` (Claude rows from before the
    experiment); `estimateFn(rows, e, paired=False)`'s sign; when not computable, every-design-helps' sign, marked `fallback`."""
    have = {_key(r) for r in pilotRows}
    rows, out = list(pilotRows) + [r for r in priorRows if _key(r) not in have], {}
    for e in effects:
        m = estimateFn(rows, e, paired=False)[1]
        out[e.id] = (dict(estimate=m, sign=_sign(m, m), fallback=False) if np.isfinite(m)
                     else dict(estimate=None, sign=e.naive, fallback=True))
    return out


def forecastNaive(effects):
    "Every-design-helps: each effect's declared naive sign (items where it does not apply are left out)."
    return {e.id: dict(sign=e.naive) for e in effects if e.naive in ('+', '-')}

## The entry point

In [ ]:
#| export
def header():
    "The code commit (git HEAD) and B3_PLAN's coopHash (b3_plan; None while it does not exist)."
    c = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip() or None
    try:
        from explRL.b3_plan import B3_PLAN
        from explRL.coop_grid import coopHash
        return dict(commit=c, b3Plan=coopHash(B3_PLAN))
    except ImportError:
        return dict(commit=c, b3Plan=None)


def items(effects, lines):
    "The forecast file's items: per effect its definition and its lines (unrepresented: pilot-only and naive only)."
    return {e.id: dict(hyp=e.hyp, measure=e.measure, fault=e.fault, N=e.N, terms=e.terms, naive=e.naive, note=e.note,
                       represented=e.represented, **({} if e.represented else dict(simulators='not represented')),
                       lines={k: v[e.id] for k, v in lines.items() if e.id in v and (e.represented or k not in SIMS)})
            for e in effects}


def forecastExperiment(exp, effects, cellFn, pilotRows, priorRows, pilotMap, reused, estimateFn=estimate, seeds=SEEDS,
                       fitFn=None, tauFn=None, refitFn=None, sims=None, targets=None, bootRows=None, nboot=2000, **extra):
    """FR-C4: theta0 fit on FIT_CELLS minus `reused`; tau; the refit on the pilot via `pilotMap` (a callable of the pilot rows, or
    refit's pilot targets); the five lines per effect, with thetas, tau, seeds, commit and B3_PLAN's hash; `extra` goes in the file.
    Injectable: `fitFn(reused)`, `tauFn(theta0, reused)`, `refitFn(theta0, tau, pilot)`; by default b3_fit's `fit(ids, sims, targets,
    drop=reused)` (ids: FIT_CELLS' ids; `reused` too), `bootTau(ids minus reused, sims, bootRows)`, `refit(theta0, tau, pilot, sims)`."""
    if None in (fitFn, tauFn, refitFn):
        from explRL.b3_fit import FIT_CELLS, bootTau, fit, refit
        ids = [c['id'] for c in FIT_CELLS]
        fitFn = fitFn or (lambda rd: fit(ids, sims, targets, drop=tuple(rd)))
        tauFn = tauFn or (lambda th, rd: bootTau([c for c in ids if c not in rd], sims, bootRows))
        refitFn = refitFn or (lambda th, tau, p: refit(th, tau, p, sims))
    pilot, th0 = pilotMap(pilotRows) if callable(pilotMap) else pilotMap, fitFn(reused)
    tau = tauFn(th0, reused)
    theta = dict(default=THETA_DEFAULT, prior=th0, pilot=refitFn(th0, tau, pilot))
    lines = {k: forecastSim(effects, cellFn, theta[t], seeds, estimateFn, nboot)
             for k, t in (('fitted', 'pilot'), ('priorOnly', 'prior'), ('unfitted', 'default'))}
    lines.update(pilotOnly=forecastPilot(effects, pilotRows, priorRows, estimateFn), naive=forecastNaive(effects))
    return dict(extra, **header(), exp=exp, seeds=list(seeds), rule=RULE, theta=theta, tau=tau,
                reused=list(reused), pilot=dict(boards=len(pilotRows), measures=pilot), lines=list(LINES),
                items=items(effects, lines))

## Files and scores

In [ ]:
#| export
def _plain(x):
    "JSON-ready: tuples to lists, numpy scalars to Python, nan to None."
    if isinstance(x, (dict, list, tuple)):
        return {str(k): _plain(v) for k, v in x.items()} if isinstance(x, dict) else [_plain(v) for v in x]
    x = x.item() if isinstance(x, np.generic) else x
    return None if isinstance(x, float) and not np.isfinite(x) else x


def writeForecast(f, path):
    "FR-C5: sorted-key JSON with a trailing newline; returns its SHA-256 (same input, same bytes)."
    b = (json.dumps(_plain(f), sort_keys=True, indent=1) + '\n').encode()
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_bytes(b)
    return hashlib.sha256(b).hexdigest()


SCORE_RULE = ("paper2's rule (P22-9): no call when the Claude 95% interval spans 0; right when it excludes 0 on the forecast's "
              'side; wrong when it excludes 0 and the forecast has the other sign or 0')


def call(sign, ci):
    "SCORE_RULE for one forecast sign and one Claude interval: 'right', 'wrong' or 'no call'."
    obs = _sign(*[_f(c) for c in ci])
    return 'no call' if obs == '0' else 'right' if sign == obs else 'wrong'


def scoreExperiment(predPath, observed, path=None):
    """FR-D1: `observed` {id: (estimate, ci)} from the experiment's report. Per item and forecaster: right, wrong or no call (a
    simulator on an unrepresented item: 0). Counts on the common set, on all items, and (P22-17) fitted and pilot-only on the items
    pilot-only computes. Writes `<exp>_score.json` beside the forecast (or at `path`); returns the score."""
    predPath = Path(predPath)
    f, res = json.loads(predPath.read_text()), {}
    for k, it in sorted(f['items'].items()):
        est, ci = observed[k]
        L = it['lines']
        r = {ln: call(L[ln]['sign'] if ln in L else '0', ci) for ln in LINES if ln in L or (ln in SIMS and not it['represented'])}
        res[k] = dict(estimate=est, ci=list(ci), observed=_sign(*[_f(c) for c in ci]), represented=it['represented'],
                      fallback=bool(L.get('pilotOnly', {}).get('fallback')), result=r)
    cnt = lambda ids, ls=LINES: {k: {o: sum(res[i]['result'].get(k) == o for i in ids) for o in ('right', 'wrong', 'no call')} for k in ls}
    common = [k for k in res if res[k]['represented']]
    computed = [k for k in res if 'pilotOnly' in res[k]['result'] and not res[k]['fallback']]
    out = dict(exp=f['exp'], forecast=predPath.name, rule=SCORE_RULE, items=res,
               common=cnt(common), all=cnt(list(res)), pilotComputed=cnt(computed, ('fitted', 'pilotOnly')),
               n=dict(common=len(common), all=len(res), pilotComputed=len(computed)))
    name = predPath.name.replace('_predictions.json', '_score.json')
    writeForecast(out, path or predPath.parent / (name if name != predPath.name else predPath.stem + '_score.json'))
    return out

## Tests

In [ ]:
# test_b3_forecasters (AC-7, FR-C1, FR-C3, FR-C4) on a fake experiment: the paired and unpaired estimates (a constant term and
# a change over team size); each forecaster's signs; pilot-only's fallback to every-design-helps' sign, marked, for an arm in
# neither the pilot nor the prior rows; an unrepresented effect has only two lines and is never simulated; the five lines
# through forecastExperiment with injected fit, tau and refit
import tempfile

rows = [dict(level=f'{a}|clean', N=N, seed=s, m=v + s % 2) for a, v in (('A', 1.0), ('B', 0.25)) for N in (8, 16) for s in (1, 2)]
dp, m = estimate(rows, Effect('x', 'h', 'm', 'clean', (8, 16), ((0.5, None), (1, 'A'), (-1, 'B'))))
assert dp == {(1, 8): 1.25, (1, 16): 1.25, (2, 8): 1.25, (2, 16): 1.25} and m == 1.25
dp, m = estimate(rows + [dict(level='A|clean', N=16, seed=1, m=5.0)], Effect('d', 'h', 'm', 'clean', 16, ((1, 'A', 16), (-1, 'A', 8))))
assert dp == {(1, 16): 3.0, (2, 16): 0.0} and m == 1.5           # seed 1: A at 16 replaced by 5.0, minus A at 8 (2.0)
assert estimate(rows, Effect('u', 'h', 'm', 'clean', 8, ((1, 'A'), (-1, 'B'))), paired=False) == ({}, 0.75)
assert np.isnan(estimate(rows, Effect('z', 'h', 'm', 'clean', 8, ((1, 'C'),)))[1])

E = (Effect('E1', 'H1', 'm', 'independent', 64, ((1, 'A'), (-1, 'B'))),               # sign follows theta's trustOne
     Effect('E2', 'H2', 'm', 'independent', 64, ((1, 'A'), (-1, 'C')), naive='-'),     # C in neither pilot nor prior
     Effect('E3', 'H3', 'm', 'independent', 64, ((1, 'D'), (-1, 'B')), represented=False))
played = []
def cellFn(arm, fault, N, seed, theta):
    played.append((arm, fault, N, seed))
    noise = np.random.default_rng(seed).normal(0, 0.05)
    t = theta.get('trustOne', 0.0)
    return dict(m=dict(A=0.3 + 0.2 * t + noise, B=0.4 + noise, C=0.2 + np.random.default_rng([seed, 1]).normal(0, 0.05),
                       D=0.0)[arm])
for th, s1 in ((dict(trustOne=1.0), '+'), (dict(trustOne=0.5), '0'), ({}, '-')):
    F = forecastSim(E, cellFn, th)
    assert set(F) == {'E1', 'E2'} and F['E1']['sign'] == s1 and F['E2']['sign'] == '+', (th, F)
assert not any(a == 'D' for a, *_ in played) and len(played) == 3 * 3 * 30          # A, B, C once per seed and setting
pilotRows = [dict(level='A|independent', N=64, seed=1890, m=0.6)]
priorRows = [dict(level=f'{a}|independent', N=64, seed=s, m=v) for a, v in (('A', 0.1), ('B', 0.4)) for s in range(1800, 1808)]
P = forecastPilot(E, pilotRows, priorRows)
assert P['E1'] == dict(estimate=0.6 - 0.4, sign='+', fallback=False)                  # A from the pilot (not the prior's 0.1), B from the prior
assert P['E2'] == dict(estimate=None, sign='-', fallback=True) and P['E3']['fallback'] and P['E3']['sign'] == '+'
assert forecastNaive(E) == {'E1': dict(sign='+'), 'E2': dict(sign='-'), 'E3': dict(sign='+')}
assert forecastNaive([Effect('t', 'h', 'm', 'clean', 8, ((1, 'A'),), naive=None)]) == {}
from explRL.b3_cells import SIM_SEEDS
assert SEEDS == SIM_SEEDS and THETA_DEFAULT == {}

calls = []
f = forecastExperiment('X', E, cellFn, pilotRows, priorRows, lambda rs: dict(boards=len(rs)), reused=('r1',),
                       fitFn=lambda rd: calls.append(rd) or dict(trustOne=0.5), tauFn=lambda th, rd: dict(trustOne=0.1),
                       refitFn=lambda th, tau, p: calls.append(p) or dict(trustOne=1.0))
assert calls == [('r1',), dict(boards=1)] and f['theta'] == dict(default={}, prior=dict(trustOne=0.5), pilot=dict(trustOne=1.0))
from explRL.b3_plan import B3_PLAN
from explRL.coop_grid import coopHash
assert f['b3Plan'] == coopHash(B3_PLAN) and len(f['commit']) == 40 and f['seeds'] == list(SEEDS) and f['reused'] == ['r1']
L = {k: v['lines'] for k, v in f['items'].items()}
assert set(L['E1']) == set(LINES) == set(L['E2']) and set(L['E3']) == {'pilotOnly', 'naive'}
assert f['items']['E3']['simulators'] == 'not represented'
assert [L['E1'][k]['sign'] for k in LINES] == ['+', '0', '-', '+', '+'] and L['E2']['pilotOnly']['fallback']
with tempfile.TemporaryDirectory() as d:
    sha = writeForecast(f, Path(d) / 'x_predictions.json')
    assert json.loads((Path(d) / 'x_predictions.json').read_text())['items']['E1']['terms'] == [[1, 'A'], [-1, 'B']]

In [ ]:
# test_b3_files (AC-9, FR-C5, FR-D1), the forecast and score files: writeForecast is byte-identical on rerun and independent of
# key order, and returns the file's SHA-256; scoreExperiment follows paper2's rule on hand-made cases (a predicted 0 on a
# conclusive outcome is wrong; an interval spanning 0 is no call; an unrepresented item is a 0 for both simulators, counted
# on all items but not on the common set) and writes <exp>_score.json beside the forecast, deterministically
ln = lambda *s: {k: dict(sign=v) for k, v in zip(LINES, s)}
fc = dict(exp='T', items=dict(
    a=dict(represented=True, lines=ln('+', '-', '0', '+', '+')),                      # Claude +: right, wrong, wrong (a 0), right, right
    b=dict(represented=True, lines=ln('+', '+', '-', '-', '+')),                      # Claude spans 0: no call for all
    c=dict(represented=True, lines=ln('-', '-', '-', '-', '+')),                      # Claude -: right x4, naive wrong
    d=dict(represented=False, lines=dict(pilotOnly=dict(sign='-', fallback=True), naive=dict(sign='+')))))
with tempfile.TemporaryDirectory() as d:
    p = Path(d) / 't_predictions.json'
    h1, b1 = writeForecast(fc, p), p.read_bytes()
    h2 = writeForecast(dict(reversed(list(fc.items()))), p)
    assert h1 == h2 == hashlib.sha256(b1).hexdigest() and p.read_bytes() == b1 and b1.endswith(b'}\n')
    obs = dict(a=(0.1, (0.02, 0.2)), b=(0.0, [-0.1, 0.1]), c=(-0.1, (-0.2, -0.01)), d=(-0.05, (-0.1, -0.01)))
    S = scoreExperiment(p, obs)
    assert (Path(d) / 't_score.json').exists() and S['items']['a']['result'] == dict(
        fitted='right', priorOnly='wrong', unfitted='wrong', pilotOnly='right', naive='right')
    assert set(S['items']['b']['result'].values()) == {'no call'}
    assert S['items']['d']['result'] == dict(fitted='wrong', priorOnly='wrong', unfitted='wrong', pilotOnly='right', naive='wrong')
    assert S['n'] == dict(common=3, all=4, pilotComputed=3)
    assert S['common']['fitted'] == {'right': 2, 'wrong': 0, 'no call': 1} and S['all']['fitted'] == {'right': 2, 'wrong': 1, 'no call': 1}
    assert S['common']['unfitted'] == {'right': 1, 'wrong': 1, 'no call': 1} and S['common']['naive'] == {'right': 1, 'wrong': 1, 'no call': 1}
    assert S['all']['pilotOnly'] == {'right': 3, 'wrong': 0, 'no call': 1} and S['pilotComputed']['pilotOnly']['right'] == 2
    b2 = (Path(d) / 't_score.json').read_bytes()
    scoreExperiment(p, obs)
    assert (Path(d) / 't_score.json').read_bytes() == b2
assert call('0', (0.1, 0.2)) == 'wrong' and call('+', (-0.1, 0.2)) == 'no call' and call('-', (-0.3, -0.2)) == 'right'